### 1. Environment Setup

#### Importing Libraries

In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn


print(f"Pandas version: {pd.__version__}")
print(f"scikit-learn version: {sklearn.__version__}")

Pandas version: 3.0.5
scikit-learn version: 1.9.0


#### Display Settings

In [2]:
# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

### 2. Data Loading and Schema Overview

#### Load Datasets

In [3]:
# === DATA PATH CONFIGURATION ===
# Local path
DATA_DIR = "./data"

# Verify data directory exists
if os.path.exists(DATA_DIR):
    print(f"✅ Data directory found: {DATA_DIR}")
    print(f"   Files: {os.listdir(DATA_DIR)}")
else:
    print(f"❌ Data directory not found: {DATA_DIR}")
    print("   Please update DATA_DIR to point to your Olist data folder")

✅ Data directory found: ./data
   Files: ['olist_sellers_dataset.csv', '.DS_Store', 'product_category_name_translation.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_products_dataset.csv']


In [4]:
# Load all Olist tables
def load_olist_data(data_dir):
    """Load all Olist CSV files into a dictionary of DataFrames."""

    tables = {
        'orders': 'olist_orders_dataset.csv',
        'order_items': 'olist_order_items_dataset.csv',
        'customers': 'olist_customers_dataset.csv',
        'products': 'olist_products_dataset.csv',
        'sellers': 'olist_sellers_dataset.csv',
        'payments': 'olist_order_payments_dataset.csv',
        'reviews': 'olist_order_reviews_dataset.csv',
        'geolocation': 'olist_geolocation_dataset.csv',
        'category_translation': 'product_category_name_translation.csv'
    }

    # Date columns to parse
    date_cols = {
        'orders': ['order_purchase_timestamp', 'order_approved_at',
                   'order_delivered_carrier_date', 'order_delivered_customer_date',
                   'order_estimated_delivery_date'],
        'order_items': ['shipping_limit_date'],
        'reviews': ['review_creation_date', 'review_answer_timestamp']
    }

    data = {}
    for name, filename in tables.items():
        filepath = os.path.join(data_dir, filename)
        parse_dates = date_cols.get(name, None)
        data[name] = pd.read_csv(filepath, parse_dates=parse_dates)
        print(f"Loaded {name}: {data[name].shape[0]:,} rows × {data[name].shape[1]} cols")

    return data

# Load data
olist = load_olist_data(DATA_DIR)

Loaded orders: 99,441 rows × 8 cols
Loaded order_items: 112,650 rows × 7 cols
Loaded customers: 99,441 rows × 5 cols
Loaded products: 32,951 rows × 9 cols
Loaded sellers: 3,095 rows × 4 cols
Loaded payments: 103,886 rows × 5 cols
Loaded reviews: 99,224 rows × 7 cols
Loaded geolocation: 1,000,163 rows × 5 cols
Loaded category_translation: 71 rows × 2 cols


#### Schema Exploration - Overview of all tables

In [5]:
# Quick schema overview
def schema_summary(data_dict):
    """Generate a summary of all tables."""
    summary = []
    for name, df in data_dict.items():
        summary.append({
            'Table': name,
            'Rows': f"{df.shape[0]:,}",
            'Columns': df.shape[1],
            'Memory (MB)': f"{df.memory_usage(deep=True).sum() / 1e6:.2f}",
            'Columns List': ', '.join(df.columns[:5]) + ('...' if len(df.columns) > 5 else '')
        })
    return pd.DataFrame(summary)

schema_summary(olist)

,Table,Rows,Columns,Memory (MB),Columns List
0,orders,"99,441",8,25.85,"order_id, customer_id, order_status, order_pur..."
1,order_items,"112,650",7,30.98,"order_id, order_item_id, product_id, seller_id..."
2,customers,"99,441",5,27.88,"customer_id, customer_unique_id, customer_zip_..."
3,products,"32,951",9,6.60,"product_id, product_category_name, product_nam..."
4,sellers,"3,095",4,0.62,"seller_id, seller_zip_code_prefix, seller_city..."
5,payments,"103,886",5,17.02,"order_id, payment_sequential, payment_type, pa..."
6,reviews,"99,224",7,29.12,"review_id, order_id, review_score, review_comm..."
7,geolocation,"1,000,163",5,135.67,"geolocation_zip_code_prefix, geolocation_lat, ..."
8,category_translation,71,2,0.01,"product_category_name, product_category_name_e..."


### 3. Orders Table Exploration and Missingness Analysis

#### Orders Table - Sample

In [6]:
# First 5 rows of Orders Table
orders = olist['orders']
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


#### Shape and Summary

In [7]:
# Orders Table Summary
print("=== Shape ===")
print(f"{orders.shape}")
print("\n")
print("=== Summary ===")
orders.info()

=== Shape ===
(99441, 8)


=== Summary ===
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


#### Order_Status Breakdown

In [8]:
print("=== Order Status Breakdown ===")
status_counts = orders['order_status'].value_counts()
status_pct = orders['order_status'].value_counts(normalize=True) * 100
print(pd.DataFrame({'Count': status_counts, 'Percentage (%)': status_pct.round(2)}))

=== Order Status Breakdown ===
              Count  Percentage (%)
order_status                       
delivered     96478           97.02
shipped        1107            1.11
canceled        625            0.63
unavailable     609            0.61
invoiced        314            0.32
processing      301            0.30
created           5            0.01
approved          2            0.00


#### Missingness Breakdown

In [9]:
print("=== Columns with Missing Value ===")
null_dates = olist['orders'].isnull().sum()
print(null_dates[null_dates > 0])

=== Columns with Missing Value ===
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64


#### Missingness: `order_delivered_customer_date`

In [10]:
# identifying rows where 'order_delivered_customer_date' is empty - returns True/False
is_blank = orders['order_delivered_customer_date'].isnull()

# filter to keep only rows where is_blank == True
blank_orders = orders[is_blank]

# group by order_status
blank_orders.groupby('order_status')['order_id'].count()

order_status
approved          2
canceled        619
created           5
delivered         8
invoiced        314
processing      301
shipped        1107
unavailable     609
Name: order_id, dtype: int64

##### Analysis - Why are there missing values in `order_delivered_customer_date`?

- 622 Orders (Status = Approved, Created, Invoiced, Processing (2 + 5 + 314 + 301)) are still in early warehouse fulfillment stage. They have not yet been dispatched yet.
- 619 Orders (Status = Canceled) are cancelled which means delivery never took place.
- 609 Orders (Status = Unavailable) are never fulfilled as the product is out of stock. No delivery took place.
- 1107 Orders (Status = Shipped) are in the midst of delivery. They have not yet reached customers
- 8 Orders (Status = Delivered) are orders with actual missing information. Need to handle these records. 

#### Missingness: `order_delivered_carrier_date`

In [11]:
# identifying rows where 'order_delivered_carrier_date' is empty - returns True/False
is_blank = orders['order_delivered_carrier_date'].isnull()

# filter to keep only rows where is_blank == True
blank_orders = orders[is_blank]

# group by order_status
blank_orders.groupby('order_status')['order_id'].count()

order_status
approved         2
canceled       550
created          5
delivered        2
invoiced       314
processing     301
unavailable    609
Name: order_id, dtype: int64

##### Analysis - Why are there missing values in `order_delivered_carrier_date`?

- 622 Orders (Status = Approved, Created, Invoiced, Processing (2 + 5 + 314 + 301)) are still in early warehouse fulfillment stage. They have not yet been dispatched yet.
- 550 Orders (Status = Canceled) are cancelled which means delivery never took place. Notice this count is different (lesser) from the count in 'order_delivered_customer_date`. A possible reason is that the cancellation came after seller dropped off the product to the carrier (but before reaching the customer).
- 609 Orders (Status = Unavailable) are never fulfilled as the product is out of stock. No delivery took place.
- 2 Orders (Status = Delivered) are orders with actual missing information. Need to handle these records. 

#### Analysis: Overall
- **Pre-dispatch Cancellations (550 Orders):** Cancelled prior to courier handover. Both `order_delivered_carrier_date` and `order_delivered_customer_date` timestamps are missing.
- **In-transit Cancellations (69 Orders):** Difference ($619 - 500 = 69$) represents packages dispatched to carrier but cancelled before reaching the buyer.
- **Active Shipments (1,107 Orders):** Dispatched to carrier but still in transit. These have `order_delivered_carrier_date` timestamps but do not have `order_delivered_customer_date` timestamps.
- **Actual Missing Information (2 to 8 Orders):** These are completed deliveries that have missing `order_delivered_carrier_date` or `order_delivered_customer_date` or both.

#### Implications and Filtering Decision
- Because target variable for **Delivery Performance** requires measuring actual lead time from purchase to delivery ($\text{Delivered Date} - \text{Purchase Date}$), orders lacking a customer delivery date cannot be included.
- **Filtering Rule:** `order_status == 'delivered'` and `order_delivered_customer_date` is not null.
- **Rows retained:** 96,470 orders (need to exclude the 8 in the missingness analysis) which is about 97% of the total orders.

### 4. Orders Items Table Exploration and Missingness Analysis

#### Orders Items Table - Sample

In [12]:
# First 5 rows of Order Items Table
order_items = olist['order_items']
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


#### Shape and Summary

In [13]:
# Order Items Table Summary
print("=== Shape ===")
print(f"{order_items.shape}")
print("\n")
print("=== Summary ===")
order_items.info()

=== Shape ===
(112650, 7)


=== Summary ===
<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  str           
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  str           
 3   seller_id            112650 non-null  str           
 4   shipping_limit_date  112650 non-null  datetime64[us]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(3)
memory usage: 6.0 MB


#### Missingness Breakdown

In [14]:
print("=== Columns with Missing Value ===")
check_missing = order_items.isnull().sum()
print(check_missing[check_missing > 0])

=== Columns with Missing Value ===
Series([], dtype: int64)


#### Check Number of Unique Orders VS Total Number of Rows

In [15]:
unique_orders = order_items['order_id'].nunique()
total_rows = len(order_items)

print("=== Comparison ===")
print(f"Unique Orders: {unique_orders:,}")
print(f"Total Rows in Table: {total_rows:,}")
print(f"Number of Orders with Multiple Items: {total_rows - unique_orders:,}")


=== Comparison ===
Unique Orders: 98,666
Total Rows in Table: 112,650
Number of Orders with Multiple Items: 13,984


#### Number of Items Per Order

In [16]:
items_per_order = order_items.groupby('order_id')['order_item_id'].count()
items_per_order.value_counts().head(5)

order_item_id
1    88863
2     7516
3     1322
4      505
5      204
Name: count, dtype: int64

#### Orders with Multiple Sellers

In [17]:
sellers_per_order = order_items.groupby('order_id')['seller_id'].nunique()
multi_seller_orders = (sellers_per_order > 1).sum()

print(f"Total Unique Sellers: {order_items['seller_id'].nunique():,}")
print(f"Orders with Multiple Sellers: {multi_seller_orders:,}")

Total Unique Sellers: 3,095
Orders with Multiple Sellers: 1,278


#### Analysis in Order Items
- **Total Unique Sellers:** 3,095 sellers across Brazil.
- **Single-Seller Orders:** 97,388 orders (98.70%) involve only 1 seller.
- **Multi-Seller Orders:** 1,278 orders (1.30%) involve multiple distinct sellers fulfilling items in a single checkout.
- **Modeling Takeaway:** When aggregating `order_items` to the order level (`order_id`), we will engineer a feature (`seller_count`) to capture this dispatch complexity.

### 5. Customers Table Exploration and Missingness Analysis

#### Customers Table - Sample

In [18]:
# First 5 rows of Customers Table
customers = olist['customers']
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


#### Shape and Summary

In [19]:
# Customers Table Summary
print("=== Shape ===")
print(f"{customers.shape}")
print("\n")
print("=== Summary ===")
customers.info()

=== Shape ===
(99441, 5)


=== Summary ===
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


#### Missingness Breakdown

In [20]:
print("=== Columns with Missing Value ===")
check_missing = customers.isnull().sum()
print(check_missing[check_missing > 0])

=== Columns with Missing Value ===
Series([], dtype: int64)


#### Key Uniqueness

In [21]:
unique_order_customers = customers['customer_id'].nunique()
unique_real_customers = customers['customer_unique_id'].nunique()

print(f"Unique customer_id (Per Order): {unique_order_customers:,}")
print(f"Unique customer_unique_id (Actual Customer): {unique_real_customers:,}")
print(f"Repeat Customer Count: {unique_order_customers - unique_real_customers:,}")

Unique customer_id (Per Order): 99,441
Unique customer_unique_id (Actual Customer): 96,096
Repeat Customer Count: 3,345


#### Validate that `customer_id` values match 1-to-1 between Orders and Customers

In [22]:
orders_cust_ids = set(orders['customer_id'])
customers_cust_ids = set(customers['customer_id'])

print(f"Total in orders: {len(orders_cust_ids):,}")
print(f"Total in customers: {len(customers_cust_ids):,}")
print(f"Matching customer_ids: {len(orders_cust_ids.intersection(customers_cust_ids)):,}")
print(f"Unmatched customer_ids: {len(orders_cust_ids.symmetric_difference(customers_cust_ids)):,}")

Total in orders: 99,441
Total in customers: 99,441
Matching customer_ids: 99,441
Unmatched customer_ids: 0


#### Analysis in Customers Table
- **Key Relationship:** `orders['customer_id']` maps 1-to-1 to `customers['customer_id']` (99,441 rows, 0 unmatched).
- **Customer Base:** 96,096 unique individuals (`customer_unique_id`), meaning ~97% of buyers are first-time/single-order customers.
- **Geographic Data:** Provides destination ZIP code prefix (`customer_zip_code_prefix`), city, and state for computing customer-seller delivery distance.

### 6. Sellers Table Exploration and Missingness Analysis

#### Sellers Table - Sample

In [23]:
# First 5 rows of Sellers Table
sellers = olist['sellers']
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


#### Shape and Summary

In [24]:
# Sellers Table Summary
print("=== Shape ===")
print(f"{sellers.shape}")
print("\n")
print("=== Summary ===")
sellers.info()

=== Shape ===
(3095, 4)


=== Summary ===
<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   int64
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: int64(1), str(3)
memory usage: 96.8 KB


#### Missingness Breakdown

In [25]:
print("=== Columns with Missing Value ===")
check_missing = sellers.isnull().sum()
print(check_missing[check_missing > 0])

=== Columns with Missing Value ===
Series([], dtype: int64)


#### Check if all sellers in Order Items table exists in Sellers table

In [26]:
# Check unique seller IDs across tables
unique_sellers_master = sellers['seller_id'].nunique()
unique_sellers_in_items = order_items['seller_id'].nunique()

# Check for unmatched IDs
unmatched_sellers = set(order_items['seller_id']) - set(sellers['seller_id'])

print(f"Total unique sellers in master list: {unique_sellers_master:,}")
print(f"Total unique sellers in order_items: {unique_sellers_in_items:,}")
print(f"Unmatched sellers (in items but not in master): {len(unmatched_sellers):,}")

Total unique sellers in master list: 3,095
Total unique sellers in order_items: 3,095
Unmatched sellers (in items but not in master): 0


#### Top Seller States and Cities

In [27]:
print("=== Top 5 Seller States ===")
seller_states = sellers['seller_state'].value_counts()
seller_states_pct = sellers['seller_state'].value_counts(normalize=True) * 100

df_s = pd.DataFrame({
    'Seller Count': seller_states,
    'Percentage (%)': seller_states_pct.round(2)
})

print(df_s.head())
print("\n")

print("=== Top 5 Seller Cities ===")
sellers['seller_city'].value_counts().head(5)

=== Top 5 Seller States ===
              Seller Count  Percentage (%)
seller_state                              
SP                    1849           59.74
PR                     349           11.28
MG                     244            7.88
SC                     190            6.14
RJ                     171            5.53


=== Top 5 Seller Cities ===


seller_city
sao paulo         694
curitiba          127
rio de janeiro     96
belo horizonte     68
ribeirao preto     52
Name: count, dtype: int64

#### Analysis in Sellers Table
- **State Clustering:** 59.74% of merchants reside in São Paulo (SP), and the top 5 states (SP, PR, MG, SC, RJ) account for 90.57% of all sellers.
- **Logistical Implication:** Because the majority of inventory originates from the South/Southeast corridor, shipping lead time and late delivery risks are strongly dictated by interstate transit distance to distant buyer locations (North/Northeast).

### 7. Products Table Exploration and Missingness Analysis

#### Products Table - Sample

In [28]:
# First 5 rows of Products Table
products = olist['products']
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.00,287.00,1.00,225.00,16.00,10.00,14.00
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.00,276.00,1.00,1000.00,30.00,18.00,20.00
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.00,250.00,1.00,154.00,18.00,9.00,15.00
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.00,261.00,1.00,371.00,26.00,4.00,26.00
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.00,402.00,4.00,625.00,20.00,17.00,13.00


#### Shape and Summary

In [29]:
# Products Table Summary
print("=== Shape ===")
print(f"{products.shape}")
print("\n")
print("=== Summary ===")
products.info()

=== Shape ===
(32951, 9)


=== Summary ===
<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 2.3 MB


#### Missingness Breakdown

In [30]:
print("=== Columns with Missing Value ===")
check_missing = products.isnull().sum()
print(check_missing[check_missing > 0])

=== Columns with Missing Value ===
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


#### Analysis in Products Table
- **Completeness of Physical Specs:** Only 2 products (0.006%) lack weight and dimensional measurements (length, height, width).
- **Missing Categories:** 610 products (1.85%) lack category names and metadata, which will be labeled as `'unknown'`.
- **Logistical Relevance:** Product weight (`product_weight_g`) and volumetric size ($\text{length} \times \text{height} \times \text{width}$) serve as physical constraints on shipping velocity and carrier handling times.

### 8. Product Category Translation Table Exploration and Missingness Analysis

#### Product Category Translation Table - Sample

In [31]:
# First 5 rows of Product Category Translation Table
category_translation = olist['category_translation']
category_translation.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


#### Shape and Summary

In [32]:
# Category Translation Table Summary
print("=== Shape ===")
print(f"{category_translation.shape}")
print("\n")
print("=== Summary ===")
category_translation.info()

=== Shape ===
(71, 2)


=== Summary ===
<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   product_category_name          71 non-null     str  
 1   product_category_name_english  71 non-null     str  
dtypes: str(2)
memory usage: 1.2 KB


#### Missingness Breakdown

In [33]:
print("=== Columns with Missing Value ===")
check_missing = category_translation.isnull().sum()
print(check_missing[check_missing > 0])

=== Columns with Missing Value ===
Series([], dtype: int64)


#### Verify Translation Coverage with Products Table

In [34]:
unique_portuguese_in_products = set(products['product_category_name'].dropna())
unique_portuguese_in_trans = set(category_translation['product_category_name'].dropna())

missing_trans = unique_portuguese_in_products - unique_portuguese_in_trans

print(f"Unique categories in products table: {len(unique_portuguese_in_products):,}")
print(f"Unique categories in translation table: {len(unique_portuguese_in_trans):,}")
print(f"Categories in products missing English: {len(missing_trans):,}")
if missing_trans:
    print(f"Unmapped categories: {missing_trans}")

Unique categories in products table: 73
Unique categories in translation table: 71
Categories in products missing English: 2
Unmapped categories: {'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'}


#### Analysis in Category Translation Table
- **Translation Coverage:** 71 out of 73 Portuguese categories map directly to English translations.
- **Unmapped Categories (2):** `'portateis_cozinha_e_preparadores_de_alimentos'` and `'pc_gamer'` are absent from the translation lookup table.
- **Handling Strategy:** A manual fallback mapping will be applied during data aggregation so that these two categories are not lost or converted to `NaN`.

### 9. Geolocation Table Exploration and Missingness Analysis

#### Geolocation Table - Sample

In [35]:
# First 5 rows of Geolocation Table
geolocation = olist['geolocation']
geolocation.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.55,-46.64,sao paulo,SP
1,1046,-23.55,-46.64,sao paulo,SP
2,1046,-23.55,-46.64,sao paulo,SP
3,1041,-23.54,-46.64,sao paulo,SP
4,1035,-23.54,-46.64,sao paulo,SP


#### Shape and Summary

In [36]:
# Geolocation Table Summary
print("=== Shape ===")
print(f"{geolocation.shape}")
print("\n")
print("=== Summary ===")
geolocation.info()

=== Shape ===
(1000163, 5)


=== Summary ===
<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  str    
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), int64(1), str(2)
memory usage: 38.2 MB


#### Missingness Breakdown

In [37]:
print("=== Columns with Missing Value ===")
check_missing = geolocation.isnull().sum()
print(check_missing[check_missing > 0])

=== Columns with Missing Value ===
Series([], dtype: int64)


#### Unique Key

In [38]:
print(f"Total Rows: {len(geolocation):,}")
print(f"Unique ZIP Code Prefixes: {geolocation['geolocation_zip_code_prefix'].nunique():,}")

Total Rows: 1,000,163
Unique ZIP Code Prefixes: 19,015


#### Minimum and Maximum Coordinates

In [39]:
print(f"=== Coordinate (Min vs Max) ===")
print(geolocation[['geolocation_lat','geolocation_lng']].describe().loc[['min', '25%', '50%', '75%', 'max']])

=== Coordinate (Min vs Max) ===
     geolocation_lat  geolocation_lng
min           -36.61          -101.47
25%           -23.60           -48.57
50%           -22.92           -46.64
75%           -19.98           -43.77
max            45.07           121.11


#### Analysis in Geolocation Table
- **Granularity:** The raw table contains 1,000,163 GPS readings across 19,015 unique 5-digit ZIP code prefixes.
- **Data Quality Anomaly Identified:** The raw dataset contains severe GPS coordinate entry errors (e.g., latitude > 40° placing points in North America/Europe, and longitude > 100° placing points in Asia). 
- **Remediation Plan:** In Section 10, we filter these coordinates using Brazil's geographic bounding box (Latitude [-34.0°, +5.5°], Longitude [-74.0°, -34.0°]) and compute 1 median coordinate pair per ZIP code prefix to build a reliable lookup dictionary.

### 10. Data Cleansing and Transformation

1. **Orders:** Filter to valid delivered orders with verified customer delivery dates (96,470 transactions).
2. **Products:** Impute missing physical measurements (weight, length, height, width) with median values and calculate volumetric size ($cm^3$).
3. **Category Translation:** Apply explicit fallback mapping for the 2 unmapped Portuguese categories (`portateis_cozinha_e_preparadores_de_alimentos` and `pc_gamer`).
4. **Geolocation:** Filter out-of-bounds GPS outliers (latitudes/longitudes outside Brazil) and deduplicate raw records into a 1-row-per-ZIP-prefix median coordinate lookup table.

#### A. Clean Orders Table: Filter for only valid delivered orders

In [40]:
# Applying Filtering Rule
# Conditions
is_delivered = orders['order_status'] == 'delivered'
has_delivery_date = orders['order_delivered_customer_date'].notna()

# Filtering
valid_orders_filter = is_delivered & has_delivery_date
orders_clean = orders[valid_orders_filter].copy()

# Summary/Count
total_count = len(orders)
retained_count = len(orders_clean)
dropped_count = total_count - retained_count

print(f"Total raw orders: {total_count:,}")
print(f"Retained delivered orders: {retained_count:,} ({retained_count/total_count*100:.2f}%)")
print(f"Filtered out: {dropped_count:,} ({ dropped_count/total_count*100:.2f}%)")

Total raw orders: 99,441
Retained delivered orders: 96,470 (97.01%)
Filtered out: 2,971 (2.99%)


#### B. Clean Products Table: Impute missing values

In [41]:
products_clean = products.copy()

products_clean['product_weight_g'] = products_clean['product_weight_g'].fillna(products_clean['product_weight_g'].median())
products_clean['product_length_cm'] = products_clean['product_length_cm'].fillna(products_clean['product_length_cm'].median())
products_clean['product_height_cm'] = products_clean['product_height_cm'].fillna(products_clean['product_height_cm'].median())
products_clean['product_width_cm'] = products_clean['product_width_cm'].fillna(products_clean['product_width_cm'].median())

products_clean['product_volume_cm3'] = (products_clean['product_length_cm'] * products_clean['product_height_cm'] * products_clean['product_width_cm'])

products_clean['product_category_name'] = products_clean['product_category_name'].fillna('unknown')
products_clean['product_name_lenght'] = products_clean['product_name_lenght'].fillna(0)
products_clean['product_description_lenght'] = products_clean['product_description_lenght'].fillna(0)
products_clean['product_photos_qty'] = products_clean['product_photos_qty'].fillna(0)

check_missing = products_clean.isnull().sum()
print(check_missing[check_missing > 0])

Series([], dtype: int64)


#### C. Clean Category Translation: Manually add missing mappings

In [42]:
# Unmapped categories: {'portateis_cozinha_e_preparadores_de_alimentos', 'pc_gamer'}
df_translation = pd.DataFrame({
    'product_category_name': [
        'portateis_cozinha_e_preparadores_de_alimentos', 
        'pc_gamer'
    ],
    'product_category_name_english': [
        'small_appliances_kitchen', 
        'pc_gamer'
    ]
})

category_translation_clean = pd.concat(
    [category_translation, df_translation], 
    ignore_index=True
)


#### D. Clean Geolocation: Coordinate Cleaning

In [43]:
valid_coords = (
    (geolocation['geolocation_lat'] >= -34.0) & (geolocation['geolocation_lat'] <= 5.5) &
    (geolocation['geolocation_lng'] >= -74.0) & (geolocation['geolocation_lng'] <= -34.0)
)

geolocation_clean = geolocation[valid_coords].copy()

geo_zip_lookup = geolocation_clean.groupby('geolocation_zip_code_prefix')[['geolocation_lat', 'geolocation_lng']].median().reset_index()

print(f"=== Clean Geolocation Lookup Table ===")
print(f"Original rows: {len(geolocation):,}")
print(f"Cleaned unique ZIP codes: {len(geo_zip_lookup):,}")

geo_zip_lookup.head()

=== Clean Geolocation Lookup Table ===
Original rows: 1,000,163
Cleaned unique ZIP codes: 19,010


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng
0,1001,-23.55,-46.63
1,1002,-23.55,-46.64
2,1003,-23.55,-46.64
3,1004,-23.55,-46.63
4,1005,-23.55,-46.64
